# Run an active-learning sampler

One sampler, one dataset, the full budget sweep. `SAMPLER` picks which one;
`VARIANTS` lists the configs to run for it. Nothing here forces a full sweep of
every sampler — one string decides.

On a Kaggle **T4 x2** session the variants are split across both GPUs, one
worker process per card. Set `PARALLEL = False` to force serial.

Per budget this writes selected indices with their per-step acquisition scores,
the probe weights, the test predictions, the metrics table, the PALM fit, a
sanity report and a run log — see `main.py`. The last cell zips all of it.

Re-running the notebook skips variants whose `_results.pt` already exists, so a
session that hits the 12-hour limit can simply be run again.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"

# Any name in sampling/specs.py. ONE sampler per notebook run:
#   random | coreset | typiclust | activeft | tcm
#   margin | entropy | badge | dropquery | uncertainty_herding | refine
#   scalpel   <- this project's method
SAMPLER = "scalpel"
SEEDS = [42]          # several seeds run back to back; run_name keeps them apart

# One dict per full budget sweep. `[{}]` runs the config file as-is, which is
# what every baseline wants. Axes worth sweeping per sampler are in
# config/config.yaml; a few examples:
#   scalpel : {"uncertainty_mode": "visual_margin"}  ablation: plain UHerding weight
#             {"cell_pooling": "rff"}                kernel mean embedding (KDE)
#             {"missing_impute": "zero"}             patches with no nucleus
#             {"consistency_weight": 0.1}            couples the two probes
#   typiclust : {"k_nn": 10}
#   tcm       : {"transition_class_multiple": 2}
#   activeft  : {"temperature": 0.1}
#   dropquery : {"dropout_ratio": 0.5}
# Sweeping a baseline's axes is baseline TUNING: either do it for every method
# or none, and say which in the report.
VARIANTS = [
    {},
    {"uncertainty_mode": "visual_margin"},
]

# Leave RUN_NAME None so each variant gets a collision-safe name of its own.
RUN_NAME = None

# Use both T4s when the session has them and there is more than one job.
PARALLEL = True

# Starting points only; the next cell searches for the real directories.
FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/features"
CELLVIT_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/cellvit_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

import main
from sampling.specs import spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_cellvit_cache, find_data_root, find_visual_cache
from utils.progress import format_duration

In [ ]:
DATA_ROOT = find_data_root()

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
base_cfg = dict(config.get("samplers", {}).get(SAMPLER, {}))
sampler_cfgs = [{**base_cfg, **overrides} for overrides in VARIANTS]
spec = spec_for(SAMPLER)

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert VARIANTS, "VARIANTS needs at least one dict (use [{}])"
assert SEEDS, "SEEDS needs at least one seed"
assert RUN_NAME is None or len(VARIANTS) * len(SEEDS) == 1, (
    "A fixed RUN_NAME with several variants/seeds makes every run overwrite the previous one"
)

# Only samplers that declare a cell view need the CellViT cache. Each seed has
# its own cache, because the train/test split is seeded.
if "cell_embeddings" in spec.needs:
    for seed in SEEDS:
        found = find_cellvit_cache(DATASET, seed, hint=CELLVIT_DIR)
        assert found is not None, (
            f"No CellViT cache for {DATASET}_seed{seed} under {CELLVIT_DIR} or "
            f"the default Kaggle input roots. Attach the extraction dataset."
        )
        CELLVIT_DIR = str(found)
    print("cellvit cache:", CELLVIT_DIR)

# The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input, which would only fail AFTER the
# whole forward pass, so fall back to a writable directory instead.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
found = find_visual_cache(DATASET, SEEDS[0], vit_name, hint=FEATURE_DIR)
if found is not None:
    FEATURE_DIR = str(found)
    print("features cache:", FEATURE_DIR)
else:
    print(f"[features] not found for {DATASET}/seed{SEEDS[0]}/{vit_name} - will extract this session.")
    print("  (a .npy with no matching manifest is rejected too -- row alignment")
    print("  cannot be verified without it.)")
    FEATURE_DIR = "/kaggle/working/features"

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"GPUs visible: {visible_gpu_count()}")
for cfg in sampler_cfgs:
    print("  config:", cfg)

In [ ]:
import time

# One job per (seed, variant). `main.run` derives the run name, so ask it for
# the same name here to know which output files a job would write.
jobs = []
skipped = []
for seed in SEEDS:
    for overrides, sampler_cfg in zip(VARIANTS, sampler_cfgs):
        name = RUN_NAME or main._default_run_name(SAMPLER, sampler_cfg)
        if seed != config.get("random_seed", 42):
            name = f"{name}_s{seed}"
        # Resume: a finished job wrote its results file. Re-running the notebook
        # after a session timeout then costs nothing for what is already done.
        if (SAVE_DIR / f"{name}_results.pt").is_file():
            skipped.append(name)
            continue
        jobs.append((name, dict(
            data_path=str(data_path),
            sampler_name=SAMPLER,
            num_classes=dataset_info["num_classes"],
            cumulative_budget=config["cumulative_budget"],
            data_descriptions=dataset_info.get("descriptions", {}),
            prompt_templates=config.get("prompt_templates", []),
            sampler_cfg=sampler_cfg,
            probe_epochs=training_cfg["probe_epochs"],
            probe_lr=training_cfg["probe_lr"],
            random_seed=seed,
            save_dir=str(SAVE_DIR),
            verbose=True,
            model_cfg=config.get("models", {}),
            feature_cache_dir=FEATURE_DIR,
            cellvit_cache_dir=CELLVIT_DIR,
            run_name=name,
            device_string="cuda:0",
        )))

if skipped:
    print(f"already finished, skipping {len(skipped)}: {', '.join(skipped)}")
print(f"to run: {len(jobs)}")
for name, _ in jobs:
    print("  ", name)

started = time.time()
workers = visible_gpu_count() if PARALLEL else 1
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
print(f"total {format_duration(time.time() - started)} | "
      f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"variants failed: {failed}"

In [ ]:
# Package the results as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape both extraction notebooks and
# run_al_baseline.ipynb use.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI. Keeping the originals beside the
# zip also doubles the download, and a session over the ~20 GB Output quota
# shows NOTHING at all, including the files that were fine.
import shutil

from utils import results_archive_stem

SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
# make_archive must not write inside the directory being archived, or it packs
# a partial copy of itself on a second run.
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = results_archive_stem(DATASET, SAMPLER, SEEDS)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")